# convaudit, Stage 2: training on converted labels (Google Colab)

**Before the first run:**
1. In Google Drive, create a folder `convaudit_stage2` and upload `convaudit.zip` into it.
2. *Runtime → Change runtime type → T4 GPU.*

**Every session:** run the cells top to bottom. Training resumes where it stopped, because checkpoints
are saved to Drive. Keep the tab open; Colab disconnects idle sessions.

There are 21 runs (7 conditions × 3 seeds). On a T4, runs with box labels take about 1.2 hours and runs with
polygon labels about 2.3 hours, roughly 40 T4 hours in all. The order is seed-major: every condition
finishes seed 0 before any starts seed 1.

**Always choose T4**, also with Colab Pro, so that every run uses the same GPU.

In [ ]:
# 1. Drive and paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # fresh view of Drive, e.g. after replacing convaudit.zip
DRIVE = '/content/drive/MyDrive/convaudit_stage2'
import os
assert os.path.exists(f'{DRIVE}/convaudit.zip'), 'Upload convaudit.zip to MyDrive/convaudit_stage2 first'

In [ ]:
# 2. Code and pinned tools (about 2 minutes)
!rm -rf /content/convaudit && unzip -q -o {DRIVE}/convaudit.zip -d /content
%cd /content/convaudit
!pip -q install ultralytics==8.4.177 supervision==0.30.9 datumaro==1.13.11 pycocotools==2.0.11 tabulate
!nvidia-smi --query-gpu=name,memory.total --format=csv
!grep -c det_reference_poly /content/convaudit/convaudit/stage2.py  # code version check: must print 3


In [ ]:
# 3. Data: built once (about 20-30 minutes) and saved to Drive; later sessions restore it in about 2 minutes
import os, subprocess
if os.path.exists(f'{DRIVE}/stage2_data.tar'):
    !tar -xf {DRIVE}/stage2_data.tar -C /content
else:
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/ann.zip
    !unzip -q -o /content/ann.zip annotations/instances_train2017.json annotations/instances_val2017.json -d /content
    !python run_stage2.py prepare --train-ann /content/annotations/instances_train2017.json --val-ann /content/annotations/instances_val2017.json --data /content/stage2_data
    !tar -cf {DRIVE}/stage2_data.tar -C /content stage2_data
# val2017 images for the final predictions (every session, about 2 minutes)
if not os.path.exists('/content/val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
    !unzip -q -o /content/val2017.zip -d /content
print(len(os.listdir('/content/stage2_data/images/train')), 'training images;',
      len(os.listdir('/content/val2017')), 'val images')

In [ ]:
# 4. Train every pending run (re-run this cell after a disconnect)
!python run_stage2.py train --data /content/stage2_data --project {DRIVE}/runs --val-images /content/val2017 --workers 2

In [ ]:
# 5. Status
!python run_stage2.py status --project {DRIVE}/runs

In [ ]:
# 6. Analysis against the original val2017 labels (when runs are done; partial results work too)
!python run_stage2.py analyze --project {DRIVE}/runs --data /content/stage2_data --out {DRIVE}/report
!head -40 {DRIVE}/report/report_stage2.md